In [4]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI18 through RI25 dataset
# see join @ https://github.com/MeganEDuffy/LCBP-interannual-EMMAs/blob/main/Notebooks/Combining-all-years-RI.ipynb
df = pd.read_csv(data_dir / "RI18-25-joined.csv")

# Load just the RI23 dataset
#df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

Hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_febros_fractions_df,
    hungerford_febros_scaler,
    hungerford_febros_pca,
    hungerford_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-02-08 00:00:00",
    end_date="2023-02-12 00:00:00",
    endmember_ids=[
                   "RI25-1110", # Baseflow
                   #"RI23-1001", # Baseflow 02/09/2023
                   "RI23-5003", # SWLD 02/15/2023
                   "RI23-5002" #  SML 02/15/2023
                    ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-1001", "RI23-5003", "RI23-5002"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-08 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-12 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties 
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "K_mg_L": 0.10,  # mg/L
    "Cu_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# Define confidence levels to process
confidence_levels = [0.70, 0.95]
results_dict = {}

for conf in confidence_levels:
    # 5. Calculate Genereux Uncertainties for the given confidence level
    uncertainty_df = up.propagate_genereux_uncertainty(
        stream_df=stream_event_df,
        em_grouped=hungerford_febros_endmembers_df,
        em_raw=em_raw_subset,
        tracers=Hungerford_tracers,
        analytical_sd=analytical_sd,
        confidence_level=conf
    )

    # 6. Merge fractions and their calculated uncertainties
    results_with_error = pd.merge(
        hungerford_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
    )

    # Sort chronologically by Datetime and reset index
    results_with_error = results_with_error.sort_values("Datetime").reset_index(drop=True)

    # Store in a dictionary for easy access in your notebook/script
    results_dict[conf] = results_with_error

    # Define dynamic filename and save to CSV in the output directory
    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Hungerford_Feb_ROS_fractions_with_uncertainty_{conf_pct}pct.csv"
    results_with_error.to_csv(output_filename, index=False)
    
    print(f"✅ Successfully saved {conf_pct}% CI results to: {output_filename}")

# Optional: Access individual dataframes if needed later in your session
results_with_error_70 = results_dict[0.70]
results_with_error_95 = results_dict[0.95]

# Preview the sorted 95% confidence results head
display_cols = ["Sample ID", "Datetime"]
uncertainty_cols_95 = [col for col in results_with_error_95.columns if "Uncertainty_95sig" in col]
fraction_cols = [col.replace("_Uncertainty_95sig", "") for col in uncertainty_cols_95]
for frac, unc in zip(fraction_cols, uncertainty_cols_95):
    display_cols.extend([frac, unc])

print("\nPreview of Chronologically Sorted 95% CI Results:")
print(results_with_error_95[display_cols].head())

✅ Successfully saved 70% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Hungerford_Feb_ROS_fractions_with_uncertainty_70pct.csv
✅ Successfully saved 95% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Hungerford_Feb_ROS_fractions_with_uncertainty_95pct.csv

Preview of Chronologically Sorted 95% CI Results:
   Sample ID            Datetime  Baseflow  Baseflow_Uncertainty_95sig  \
0  RI23-1001 2023-02-09 13:40:00  0.875601                    0.760915   
1  RI23-1002 2023-02-09 16:00:00  0.811964                    0.818667   
2  RI23-1003 2023-02-09 22:00:00       NaN                    0.702839   
3  RI23-1005 2023-02-10 10:00:00  0.216741                    0.458646   
4  RI23-1007 2023-02-10 16:00:00  0.416329                    0.306862   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_95sig  \
0        1.243986e-01                          3.703617e-09   
1        1.880355e

In [5]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

Hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_martherm_fractions_df,
    hungerford_martherm_scaler,
    hungerford_martherm_pca,
    hungerford_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
                   #"RI23-1001", # Baseflow 02/09/2023
                   "RI23-1035", # 03/22/2023 Baseflow
                   #"RI23-5008", # Mark's well 2023
                   #"RI20-HB-DGW-Mark-1" # Mark's well 2020
                   #"RI22-0858", # Mark's well 03/17/2022
                   #"RI22-0864",
                   #"RI23-1085", #Baseflow 03/31/2023 removed for now
                   "RI23-5007", # SWLD
                   "RI23-1061"
                    ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin([["RI23-1035", "RI23-5007", "RI23-1061"]])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "K_mg_L": 0.10,  # mg/L
    "Cu_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# Define confidence levels to process
confidence_levels = [0.70, 0.95]
results_dict = {}

for conf in confidence_levels:
    # 5. Calculate Genereux Uncertainties for the given confidence level
    uncertainty_df = up.propagate_genereux_uncertainty(
        stream_df=stream_event_df,
        em_grouped=hungerford_martherm_endmembers_df,
        em_raw=em_raw_subset,
        tracers=Hungerford_tracers,
        analytical_sd=analytical_sd,
        confidence_level=conf
    )

    # 6. Merge fractions and their calculated uncertainties
    results_with_error = pd.merge(
        hungerford_martherm_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
    )

    # Sort chronologically by Datetime and reset index
    results_with_error = results_with_error.sort_values("Datetime").reset_index(drop=True)

    # Store in a dictionary for easy access in your notebook/script
    results_dict[conf] = results_with_error

    # Define dynamic filename and save to CSV in the output directory
    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Hungerford_Mar_therm_fractions_with_uncertainty_{conf_pct}pct.csv"
    results_with_error.to_csv(output_filename, index=False)
    
    print(f"✅ Successfully saved {conf_pct}% CI results to: {output_filename}")

# Optional: Access individual dataframes if needed later in your session
results_with_error_70 = results_dict[0.70]
results_with_error_95 = results_dict[0.95]

# Preview the sorted 95% confidence results head
display_cols = ["Sample ID", "Datetime"]
uncertainty_cols_95 = [col for col in results_with_error_95.columns if "Uncertainty_95sig" in col]
fraction_cols = [col.replace("_Uncertainty_95sig", "") for col in uncertainty_cols_95]
for frac, unc in zip(fraction_cols, uncertainty_cols_95):
    display_cols.extend([frac, unc])

print("\nPreview of Chronologically Sorted 95% CI Results:")
print(results_with_error_95[display_cols].head())

✅ Successfully saved 70% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Hungerford_Mar_therm_fractions_with_uncertainty_70pct.csv
✅ Successfully saved 95% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Hungerford_Mar_therm_fractions_with_uncertainty_95pct.csv

Preview of Chronologically Sorted 95% CI Results:
   Sample ID            Datetime  Baseflow  Baseflow_Uncertainty_95sig  \
0  RI23-1035 2023-03-22 15:00:00  1.000000                    0.677134   
1  RI23-1036 2023-03-22 18:00:00  0.930323                    1.109767   
2  RI23-1037 2023-03-23 00:00:00  0.666576                    0.680186   
3  RI23-1038 2023-03-23 06:00:00  0.676390                    1.108000   
4  RI23-1042 2023-03-23 12:00:00  0.730678                    1.155190   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_95sig  \
0        7.160939e-15                              0.360217   
1        6.967

In [6]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_fmelt_fractions_df,
    hungerford_fmelt_scaler,
    hungerford_fmelt_pca,
    hungerford_fmelt_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-30 00:00:00",
    end_date="2023-04-04 00:00:00",
    endmember_ids=[
                   "RI23-1001", # Baseflow 02/09/2023
                   #"RI23-1035", # 03/22/2023 Baseflow
                   #"RI23-1085", # Baseflow 03/31/23
                   #"RI23-5008", # Groundwater (Mark's well) 03/16/2023
                   #"RI20-HB-DGW-Mark-1" # Mark's well 2020
                   #"RI22-0858", # Mark's well 03/17/2022
                   #"RI22-0864", # Mark's well 07/27/2022
                   #"RI23-5014", # Soil water lysimeter dry 04/12/23
                   "RI23-5013", # Soil water lysimeter wet 04/12/23
                   "RI23-5017", # Snowmelt lysimeter 04/12/23
                   "RI23-1061" # Snowmelt lysimeter 03/28/23
                    ],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-1001", "RI23-5013", "RI23-5017", "RI23-1061"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-30 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-04-04 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "K_mg_L": 0.10,  # mg/L
    "Cu_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# Define confidence levels to process
confidence_levels = [0.70, 0.95]
results_dict = {}

for conf in confidence_levels:
    # 5. Calculate Genereux Uncertainties for the given confidence level
    uncertainty_df = up.propagate_genereux_uncertainty(
        stream_df=stream_event_df,
        em_grouped=hungerford_fmelt_endmembers_df,
        em_raw=em_raw_subset,
        tracers=Hungerford_tracers,
        analytical_sd=analytical_sd,
        confidence_level=conf
    )

    # 6. Merge fractions and their calculated uncertainties
    results_with_error = pd.merge(
        hungerford_fmelt_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
    )

    # Sort chronologically by Datetime and reset index
    results_with_error = results_with_error.sort_values("Datetime").reset_index(drop=True)

    # Store in a dictionary for easy access in your notebook/script
    results_dict[conf] = results_with_error

    # Define dynamic filename and save to CSV in the output directory
    conf_pct = int(conf * 100)
    output_filename = output_dir / f"Hungerford_final_melt_fractions_with_uncertainty_{conf_pct}pct.csv"
    results_with_error.to_csv(output_filename, index=False)
    
    print(f"✅ Successfully saved {conf_pct}% CI results to: {output_filename}")

# Optional: Access individual dataframes if needed later in your session
results_with_error_70 = results_dict[0.70]
results_with_error_95 = results_dict[0.95]

# Preview the sorted 95% confidence results head
display_cols = ["Sample ID", "Datetime"]
uncertainty_cols_95 = [col for col in results_with_error_95.columns if "Uncertainty_95sig" in col]
fraction_cols = [col.replace("_Uncertainty_95sig", "") for col in uncertainty_cols_95]
for frac, unc in zip(fraction_cols, uncertainty_cols_95):
    display_cols.extend([frac, unc])

print("\nPreview of Chronologically Sorted 95% CI Results:")
print(results_with_error_95[display_cols].head())

✅ Successfully saved 70% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Hungerford_final_melt_fractions_with_uncertainty_70pct.csv
✅ Successfully saved 95% CI results to: /home/millieginty/OneDrive/git-repos/LCBP-interannual-EMMAs/Output/EMMA-uncertainty/Hungerford_final_melt_fractions_with_uncertainty_95pct.csv

Preview of Chronologically Sorted 95% CI Results:
   Sample ID            Datetime  Baseflow  Baseflow_Uncertainty_95sig  \
0  RI23-1085 2023-03-31 14:00:00  0.684559                    0.668958   
1  RI23-1086 2023-03-31 20:00:00  0.684772                    0.714531   
2  RI23-1087 2023-04-01 02:00:00  0.686955                    0.718421   
3  RI23-1088 2023-04-01 08:00:00  0.703818                    0.669518   
4  RI23-1089 2023-04-01 14:00:00  0.721186                    0.691663   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_95sig  \
0        6.364304e-02                          3.848862e-08   
1        1.0